# 02a — Chợ Tốt Cleaning Pipeline

Notebook này chạy pipeline reproducible để biến raw Chợ Tốt laptop listings thành clean dataset sẵn sàng cho feature engineering/modeling.

Logic cleaning nằm trong `src/clean/chotot_cleaning.py`; notebook này chỉ import module, chạy pipeline, và hiển thị audit summary.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "Laptop-Price-Prediction":
    candidates = [p for p in PROJECT_ROOT.parents if p.name == "Laptop-Price-Prediction"]
    PROJECT_ROOT = candidates[0] if candidates else PROJECT_ROOT

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clean.chotot_cleaning import CONFIG, run_pipeline

CONFIG


{'raw_path': WindowsPath('Y:/Python/Laptop-Price-Prediction/data/raw/chotot_laptop_data.csv'),
 'issues_path': WindowsPath('Y:/Python/Laptop-Price-Prediction/docs/chotot_issues_list.csv'),
 'output_dir': WindowsPath('Y:/Python/Laptop-Price-Prediction/data/intern'),
 'cleaned_path': WindowsPath('Y:/Python/Laptop-Price-Prediction/data/intern/chotot_cleaned.csv'),
 'report_path': WindowsPath('Y:/Python/Laptop-Price-Prediction/docs/chotot_cleaning_report.csv'),
 'issue_action_plan_path': WindowsPath('Y:/Python/Laptop-Price-Prediction/docs/chotot_issue_action_plan.csv'),
 'log_path': WindowsPath('Y:/Python/Laptop-Price-Prediction/docs/chotot_cleaning_log.json'),
 'price_min': 1000000,
 'price_max': 60000000,
 'rare_count_threshold': 30,
 'valid_ram_levels': {2, 4, 6, 8, 12, 16, 24, 32, 36, 48, 64},
 'title_valid_ram_levels': {2, 4, 6, 8, 12, 16, 24, 32, 36, 48, 64, 96, 128},
 'screen_min': 8,
 'screen_max': 22,
 'price_bins': [0, 5000000, 10000000, 20000000, inf],
 'price_labels': ['low', '

## Run Pipeline

Các bước trong `run_pipeline`: load raw + issue list, build `issue_action_plan`, normalize text, clean price, extract brand/model/specs, flag missing/outliers/inconsistencies, group low-count categories, drop only the safe constant column, validate, and save outputs.

In [2]:
artifacts = run_pipeline(CONFIG)

Raw schema:
url                    object
price                  object
title                  object
Hãng                   object
Dòng máy               object
Tình trạng             object
Chính sách bảo hành    object
Kích cỡ màn hình       object
Bộ vi xử lý            object
RAM                    object
Card màn hình          object
Ổ cứng                 object
Xuất xứ                object
Loại ổ cứng            object
Thông tin sử dụng      object
dtype: object

Raw shape: 5,866 rows x 15 columns

Sample records:
                                                                             url         price                                             title  Hãng       Dòng máy                  Tình trạng Chính sách bảo hành Kích cỡ màn hình       Bộ vi xử lý    RAM Card màn hình    Ổ cứng        Xuất xứ Loại ổ cứng Thông tin sử dụng
0      https://www.chotot.com/mua-ban-quan-tan-binh-tp-ho-chi-minh/121607693.htm   9.990.000 đ          Laptop 2in1 Dell 7420 I7 1185G7/16G/256G  

## Quick Audit

In [3]:
print('Cleaned shape:', artifacts.cleaned.shape)
print('\nIssue status counts:')
print(artifacts.issue_action_plan['resolution_status'].value_counts().to_string())
print('\nImportant rule counts:')
for key, value in artifacts.log['important_rule_counts'].items():
    print(f'- {key}: {value}')

artifacts.cleaned.head()

Cleaned shape: (5866, 60)

Issue status counts:
resolution_status
needs_manual_review    41
resolved               29
partially_resolved     26

Important rule counts:
- price_missing: 1
- price_outlier: 167
- ram_suspicious: 23
- potential_dedicated_gpu: 40
- storage_type_imputed: 11
- new_low_price: 25
- repair_mismatch: 27


,url,price,title,Hãng,Dòng máy,Tình trạng,Chính sách bảo hành,Kích cỡ màn hình,Bộ vi xử lý,RAM,...,storage_type_clean,was_storage_type_imputed,condition_clean,new_low_price,repair_mismatch,condition_warranty_inconsistent,brand_grouped,model_grouped,brand_is_rare,model_is_rare
0,https://www.chotot.com/mua-ban-quan-tan-binh-t...,9.990.000 đ,Laptop 2in1 Dell 7420 I7 1185G7/16G/256G,Dell,Latitude,Đã sử dụng (chưa sửa chữa),>12 tháng,13 - 14.9 inch,Intel Core i7,16 GB,...,SSD,False,Đã sử dụng (chưa sửa chữa),False,False,False,Dell,Latitude,False,False
1,https://www.chotot.com/mua-ban-thanh-pho-thu-d...,4.500.000 đ,Acer Aspire A315-58 i3-1115G4,Acer,Aspire A3,Đã sử dụng (chưa sửa chữa),Bảo hành hãng,15 - 16.9 inch,Intel Core i3,8 GB,...,SSD,False,Đã sử dụng (chưa sửa chữa),False,False,False,Acer,Other,False,True
2,https://www.chotot.com/mua-ban-quan-ha-dong-ha...,3.500.000 đ,Thanh lý Laptop HP Elite x2 1012G1 cảm ứng 2in1,HP,Elite X2,Đã sử dụng (chưa sửa chữa),Bảo hành hãng,11 - 12.9 inch,Intel Core i7,8 GB,...,SSD,False,Đã sử dụng (chưa sửa chữa),False,False,False,HP,Other,False,True
3,https://www.chotot.com/mua-ban-quan-7-tp-ho-ch...,37.500.000 đ,Dell Pro Rugged RB14250 Ultra 7-165U 32GB 512GB,Dell,Pro Rugged 14,Đã sử dụng (chưa sửa chữa),>12 tháng,13 - 14.9 inch,Intel Core Ultra,32 GB,...,SSD,False,Đã sử dụng (chưa sửa chữa),False,False,False,Dell,Other,False,True
4,https://www.chotot.com/mua-ban-quan-12-tp-ho-c...,7.990.000 đ,HP Spectre x360 13-ae015dx i7 cảm ứng gập 360 độ,HP,Spectre,Đã sử dụng (chưa sửa chữa),3 tháng,13 - 14.9 inch,<NA>,16 GB,...,SSD,False,Đã sử dụng (chưa sửa chữa),False,False,False,HP,Other,False,True


In [4]:
artifacts.issue_action_plan.head(15)

,issue_key,severity,affected_column,issue_group,issue_description,proposed_action,need_note_lookup,final_action,resolution_status
0,price,high,price,outliers,"Giá có phân phối lệch mạnh (right-skewed), tập...",Áp dụng log transform khi modeling; kiểm tra v...,yes,Leave for modeling; document as modeling decis...,needs_manual_review
1,price_outliers,high,price,outliers,Tồn tại các giá trị bất thường (quá thấp hoặc ...,Dùng IQR hoặc percentile để detect; xử lý bằng...,yes,Create explicit flags; do not drop uncertain r...,partially_resolved
2,price_market_bias,medium,price,price_related_issues,"Giá phản ánh thị trường Việt Nam (second-hand,...",Không suy rộng kết luận ra thị trường quốc tế;...,no,Document local-market scope; no row-level clea...,needs_manual_review
3,title,high,title,missing_null_empty_values,"Text tự do, giàu thông tin nhưng không đồng đề...",Giữ làm metadata; extract feature có kiểm soát...,yes,Create missing indicators and conservative fal...,partially_resolved
4,title_parsing,low,title,text_normalization_specs_parsing,"Có ambiguity cao khi extract, đặc biệt RAM vs ...",Dùng parsing có ngữ cảnh (RAM keyword + valid ...,no,Documented; no deterministic cleaning required.,needs_manual_review
5,title_recovery,high,title,missing_null_empty_values,"Khả năng fill missing thấp: CPU ~12%, RAM ~13%...",Chỉ dùng title như fallback; kết hợp imputatio...,yes,Apply conservative parser; never overwrite str...,partially_resolved
6,title_consistency,low,title,missing_null_empty_values,Có conflict đáng kể với structured columns (RA...,Không dùng title để overwrite dữ liệu có sẵn; ...,no,Create missing indicators and conservative fal...,partially_resolved
7,title_structure,low,title,text_normalization_specs_parsing,"Khoảng ~10% title có pattern dạng ""X/Y"" (8/256...",Khai thác bằng parser riêng để tách RAM/Storag...,yes,Apply conservative parser; never overwrite str...,partially_resolved
8,title_cpu_coverage,low,title,text_normalization_specs_parsing,"Regex CPU chưa cover hết pattern (Core2 Duo, N...",Chấp nhận coverage không hoàn toàn; không rely...,yes,Documented; no deterministic cleaning required.,needs_manual_review
9,brand_price_segment,medium,price,price_related_issues,Phân bố giá theo brand không đồng đều; một số ...,Giữ brand làm feature chính; có thể tạo thêm f...,no,Preserve as feature or helper signal in cleane...,resolved


In [5]:
artifacts.report.head(30)

,section,metric,value
0,shape,row_count_before,5866
1,shape,row_count_after,5866
2,shape,column_count_before,15
3,shape,column_count_after,60
4,duplicates,full_duplicate_removed,0
5,duplicates,url_duplicate_removed,0
6,price,price_missing_flagged,1
7,price,price_outlier_flagged,167
8,specs,ram_suspicious_flagged,23
9,specs,storage_missing_all_flagged,342
